# 14 — Product Comparison Smoke Test

هدف این notebook فقط smoke test فاز مقایسه است:

- انتخاب ۲ محصول مشخص
- metadata مستقیم محصول
- review retrieval جدا برای هر محصول
- structured grounded comparison
- citation ownership validation
- latency / token telemetry

هیچ `def` یا `class` داخل notebook تعریف نمی‌شود.

In [1]:
import os
from pathlib import Path
import sys

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

from src.rag.config import load_config
from src.rag.generation import OpenAIJSONGenerator
from src.rag.pipeline import ProductComparisonPipeline
from src.rag.runtime import load_retrieval_stack

In [2]:
rag_config = load_config(
    PROJECT_ROOT / "configs" / "rag.yaml"
)

qa_config = load_config(
    PROJECT_ROOT / "configs" / "qa.yaml"
)

comparison_config = load_config(
    PROJECT_ROOT / "configs" / "comparison.yaml"
)["comparison"]

retrieval = load_retrieval_stack(
    project_root=PROJECT_ROOT,
    rag_config=rag_config,
)

product_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "products_search.parquet"
)

product_documents = pd.read_parquet(
    product_path
)

print("Products:", len(product_documents))
print("Comments:", len(retrieval.documents))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Products: 948352
Comments: 6153060


In [3]:
generation_config = qa_config["generation"]

generator = OpenAIJSONGenerator(
    api_key=os.environ["METIS_API_KEY"],
    base_url=os.environ["METIS_BASE_URL"],
    model=generation_config["model"],
    input_cost_per_million=(
        generation_config.get(
            "input_cost_per_million"
        )
    ),
    output_cost_per_million=(
        generation_config.get(
            "output_cost_per_million"
        )
    ),
)

comparison = ProductComparisonPipeline(
    product_documents=product_documents,
    review_retriever=retrieval.hybrid,
    review_documents=retrieval.documents,
    generator=generator,
    reviews_per_product=(
        comparison_config["reviews_per_product"]
    ),
    min_products=(
        comparison_config["min_products"]
    ),
    max_products=(
        comparison_config["max_products"]
    ),
    max_context_chars=(
        comparison_config["max_context_chars"]
    ),
    max_chars_per_review=(
        comparison_config["max_chars_per_review"]
    ),
)

## سناریوی smoke test

دو ضدآفتاب از نمونه‌های قبلی پروژه را مقایسه می‌کنیم. اگر در نسخه‌ی
دیتاست شما یکی از IDها موجود نبود، فقط `PRODUCT_IDS` را با دو شناسه‌ی
موجود عوض کنید.

In [4]:
PRODUCT_IDS = [6733478, 6283256]

COMPARISON_QUERY = (
    "برای پوست چرب کدام گزینه مناسب‌تر است؟ "
    "از نظر حس روی پوست و تجربه‌ی جوش یا سنگینی هم مقایسه کن."
)

selected = product_documents[
    product_documents["id"]
    .astype(int)
    .isin(PRODUCT_IDS)
][
    [
        "id",
        "title_fa",
        "Brand",
        "Category2",
        "Price",
        "Rate",
        "Rate_cnt",
    ]
]

display(selected)

if len(selected) != len(PRODUCT_IDS):
    missing = sorted(
        set(PRODUCT_IDS)
        - set(
            selected["id"]
            .astype(int)
            .tolist()
        )
    )
    raise ValueError(
        f"Product IDs not found: {missing}"
    )

,id,title_fa,Brand,Category2,Price,Rate,Rate_cnt
299140,6283256,کرم ضد آفتاب ژیناژن SPF50 مدل 02 مناسب پوست ها...,ژیناژن,کرم ضد آفتاب,2376800.0,78,82
324380,6733478,فلوئید ضد آفتاب بی رنگ الارو SPF50 مدل Ultra L...,الارو,کرم ضد آفتاب,4141400.0,88,620


In [5]:
result = comparison.compare(
    product_ids=PRODUCT_IDS,
    query=COMPARISON_QUERY,
)

print("Citation valid:", result["citation_valid"])
print("Citation repaired:", result["citation_repaired"])
print("Confidence:", result["confidence"])
print("Insufficient evidence:", result["insufficient_evidence"])
print("Overall winner:", result["overall_winner_product_id"])
print()
print("Summary:")
print(result["summary"])
print()
print("Recommendation:")
print(result["overall_recommendation"])

Citation valid: True
Citation repaired: False
Confidence: medium
Insufficient evidence: False
Overall winner: 6283256

Summary:
هر دو محصول در عنوان و نظر کاربران برای پوست چرب مطرح شده‌اند. برای حس سبک روی پوست، الارو شواهد مستقیم‌تری دارد اما درباره افزایش چربی هم گزارش متناقض وجود دارد. درباره جوش، شواهد مستقیمی برای هیچ‌کدام ارائه نشده است.

Recommendation:
برای پوست چرب، ژیناژن بر اساس نظرات موجود انتخاب مطمئن‌تری به نظر می‌رسد، چون بازخوردهای تناسب آن با پوست چرب مثبت‌تر و بدون مخالفت مستقیم است. اگر اولویت شما حس بسیار سبک است، الارو شواهد مستقیم دارد، اما احتمال احساس چربی بیشتر در برخی افراد نیز گزارش شده است. درباره جوش نمی‌توان بین آن‌ها نتیجه‌گیری کرد.


In [6]:
criteria_rows = []

for criterion in result["criteria"]:
    winner = criterion.get(
        "winner_product_id"
    )

    for assessment in criterion[
        "assessments"
    ]:
        criteria_rows.append(
            {
                "criterion": criterion["name"],
                "product_id": assessment["product_id"],
                "stance": assessment["stance"],
                "assessment": assessment["text"],
                "evidence_ids": assessment["evidence_ids"],
                "criterion_winner": winner,
                "winner_reason": criterion.get(
                    "winner_reason",
                    "",
                ),
            }
        )

criteria_df = pd.DataFrame(
    criteria_rows
)

display(criteria_df)

,criterion,product_id,stance,assessment,evidence_ids,criterion_winner,winner_reason
0,تناسب با پوست چرب,6733478,mixed,عنوان محصول آن را مناسب پوست چرب و مختلط معرفی...,"[41863872, 49492180]",6283256.0,بازخوردهای موجود درباره تناسب ژیناژن با پوست چ...
1,تناسب با پوست چرب,6283256,positive,عنوان محصول برای پوست چرب است و سه نظر موجود ن...,"[39654770, 35623820, 41551497]",6283256.0,بازخوردهای موجود درباره تناسب ژیناژن با پوست چ...
2,حس روی پوست و سنگینی,6733478,mixed,یک کاربر از سبک بودن و نبود حسِ محصول روی پوست...,"[41863872, 50710124]",NaN,برای الارو تجربه سبک بودن گزارش شده، اما گزارش...
3,حس روی پوست و سنگینی,6283256,unknown,در نظرات ارائه‌شده، توضیح مستقیمی درباره سبکی،...,[],NaN,برای الارو تجربه سبک بودن گزارش شده، اما گزارش...
4,تجربه جوش,6733478,unknown,در نظرات ارائه‌شده اشاره مستقیمی به ایجاد جوش ...,[],NaN,شواهدی درباره تجربه جوش برای هیچ‌یک از دو محصو...
5,تجربه جوش,6283256,unknown,در نظرات ارائه‌شده اشاره مستقیمی به ایجاد جوش ...,[],NaN,شواهدی درباره تجربه جوش برای هیچ‌یک از دو محصو...


In [7]:
print(
    "Retrieved review IDs by product:",
    result[
        "retrieved_review_ids_by_product"
    ],
)

print(
    "Cited evidence IDs by product:",
    result[
        "evidence_ids_by_product"
    ],
)

evidence_columns = [
    column
    for column in [
        "id",
        "product_id",
        "rate",
        "body",
        "score",
    ]
    if column
    in result[
        "evidence_documents"
    ].columns
]

display(
    result[
        "evidence_documents"
    ][
        evidence_columns
    ]
)

Retrieved review IDs by product: {6733478: [41863872, 49492180, 50710124], 6283256: [35623820, 39654770, 41551497]}
Cited evidence IDs by product: {6733478: [41863872, 49492180, 50710124], 6283256: [39654770, 35623820, 41551497]}


,id,product_id,rate,body,score
0,41863872,6733478,4.0,برای پوست‌های چرب مناسبه. صورت سنگین نمیشه و ا...,0.877795
1,49492180,6733478,3.0,مناسب پوست چرب نیست,0.488543
2,50710124,6733478,4.0,ب توصیه بقیه تازه گرفتم و استفاده میکنم برای پ...,0.769948
3,39654770,6283256,5.0,مناسب برای پوست های چرب,1.000000
4,35623820,6283256,5.0,عالیه برای پوست های چرب,0.631274
5,41551497,6283256,3.0,پوشش خیلی خوبی داره و برای پوست چرب مناسبه,0.484225


In [8]:
telemetry = result["telemetry"]

print(
    "Review retrieval:",
    round(
        telemetry[
            "review_retrieval_latency_ms"
        ],
        1,
    ),
    "ms",
)

print(
    "Generation:",
    round(
        telemetry[
            "generation_latency_ms"
        ] / 1000,
        2,
    ),
    "sec",
)

print(
    "Total:",
    round(
        telemetry[
            "total_latency_ms"
        ] / 1000,
        2,
    ),
    "sec",
)

print(
    "Tokens:",
    telemetry["total_tokens"],
)

print(
    "Model:",
    telemetry["model"],
)

print(
    "Retrieved reviews:",
    telemetry[
        "retrieved_review_count"
    ],
)

Review retrieval: 1450.9 ms
Generation: 11.6 sec
Total: 13.09 sec
Tokens: 1556
Model: gpt-5.6-terra
Retrieved reviews: 6
